In [17]:
import pandas as pd
import requests
import json
import numpy as np

def extract_data():
    url = "https://remotive.com/api/remote-jobs"
    try:
        response = requests.get(url)
        response.raise_for_status
        json_file = response.json()

        with open(r"raw/remotive_data.json",'w') as file:
            json.dump(json_file,file,indent=4)
            return json_file
    except Exception as e:
        print(f'Erro ao buscar os dados:{e}')

data = extract_data()



In [ ]:
# print(type(data))
# print(data.keys())
# print(data.values()

# # Exploração e definição do dataframe:
dados = data["jobs"]


def trat_dados_origem(dados):
    """Realiza o tratamento geral da base de dados. como mundança de tipos e padronização de valores."""
    df = pd.DataFrame(dados)
    df_origem = df.copy()
    df_origem.columns

    columns_drop = ['company_logo_url','company_logo','url','description']
    df_origem = df_origem.drop(columns=columns_drop)

     # definindo moeda
    df_origem["currency"] = df_origem["salary"].apply(lambda x: "USD" if isinstance(x, str) and "$" in x else np.nan)

    # tratando inconsistencias no salario 
    df_origem["salary"] = (
        df_origem["salary"]
        .str.replace("OTE", "", regex=False)
        .str.strip()
    )
    
    # separando periodo
    hour = df_origem["salary"].str.contains(r'\b(hour|hr)\b',case=False,na=False)
    df_origem["period"] = np.where(hour,"hour","year")
    
    # separando faixa salarial:
    faixa = df_origem["salary"].str.split(r'\s*-\s*', n=1, expand=True)
    parte_min = faixa[0]
    parte_max = faixa[1]

    def extrai_valor(serie):
        """Faz a extração dos números na string, para determinar os valores minimo e maximo da faixa salarial."""
        # pega número (com , ou . decimal) + 'k' opcional
        num = serie.str.extract(r'(\d+(?:[.,]\d+)?)\s*([kK])?')
        valor = num[0].str.replace(',', '.', regex=False).astype(float)
        valor = np.where(num[1].notna(), valor * 1000, valor)
        return valor

    df_origem["salary_min"] = extrai_valor(parte_min)
    df_origem["salary_max"] = extrai_valor(parte_max)
    
    df_origem["salary_max"] = df_origem["salary_max"].fillna(df_origem["salary_min"])


    # tratando coluna de data:
    df_origem["publication_date"] = pd.to_datetime(df_origem["publication_date"],errors='coerce')


    return df_origem


# Explosão da lista de habilidades da vaga, pra posterior manipulação:

df_exploded_skills = trat_dados_origem(dados).explode("tags",ignore_index=True)



# Modelagem dos dados:

# Tratamento

def df_empresa(df_origem):
    dim_empresa = (
        df_origem[["company_name"]]
        .drop_duplicates()
        .reset_index(drop=True))
    return dim_empresa


def df_skill(df_exploded):

    dim_skill = (
    df_exploded[["tags"]]
    .drop_duplicates()
    .reset_index(drop=True)

)
    dim_skill = dim_skill.rename(columns={"tags":"skill"})
    dim_skill["skill"] = dim_skill["skill"].fillna("Não especificado")
    return dim_skill

def df_categoria(df_origem):
    dim_categoria = (
        df_origem[["category"]]
        .drop_duplicates()
        .reset_index(drop=True)

    )
    return dim_categoria 


df_origem = trat_dados_origem(dados)

df_origem.sample(10)






C:\Users\ys100\AppData\Local\Temp\ipykernel_4916\4000600849.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hour = df_origem["salary"].str.contains(r'\b(hour|hr)\b',case=False,na=False)
C:\Users\ys100\AppData\Local\Temp\ipykernel_4916\4000600849.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hour = df_origem["salary"].str.contains(r'\b(hour|hr)\b',case=False,na=False)


,id,title,company_name,category,tags,job_type,publication_date,candidate_required_location,salary,currency,period,salary_min,salary_max
26,1749306,Freelance Copywriter,Coalition Technologies,Writing,"[accounting, excel, research, data analysis, b...",freelance,2026-07-02 20:01:13,Worldwide,$20k -$35k,USD,year,20000.0,35000.0
39,2090998,Senior Quality Engineer (Campinas),LawnStarter,Quality Assurance,"[backend, php, react, sql, AI/ML, automation, ...",full_time,2026-06-23 08:23:50,Brazil,$45k - $65k,USD,year,45000.0,65000.0
34,2090984,Senior Product Manager,Lemon.io,Product Management,"[.Net, android, azure, C, C#, C++, data scienc...",full_time,2026-06-24 09:57:46,"Americas, Europe, Asia, Oceania",,NaN,year,NaN,NaN
17,2091053,Staff Product Engineer (Campinas),LawnStarter,Product Management,"[AWS, backend, frontend, php, react, security,...",full_time,2026-07-09 14:28:36,Brazil,$80k - $100k,USD,year,80000.0,100000.0
0,2091069,Patient Care Specialist,STATLINX,Medical,"[research, insurance]",full_time,2026-07-16 13:28:02,USA,$36k,USD,year,36000.0,36000.0
13,2091049,"Staff Software Engineer, Product (Campinas)",LawnStarter,Software Development,"[AWS, backend, frontend, php, react, security,...",full_time,2026-07-09 14:39:55,Brazil,$80k - $100k,USD,year,80000.0,100000.0
41,2090887,Mid/Senior AI Cinematic Video Editor,EverAI,Artificial Intelligence,"[excel, video, AI/ML, customer acquisition, gr...",full_time,2026-06-19 19:46:09,Worldwide,,NaN,year,NaN,NaN
20,2091048,Product Sales Specialist - Pet Health,Tribe Wellness,Sales,"[excel, salesforce, video, spark]",full_time,2026-07-09 08:01:56,"USA, CST (UTC-6)",$55k - $100k,USD,year,55000.0,100000.0
12,2091050,"Staff Software Engineer, Product (Mexico City)",LawnStarter,Software Development,"[AWS, backend, frontend, php, react, security,...",full_time,2026-07-09 14:40:42,Mexico,$80k - $100k,USD,year,80000.0,100000.0
6,2091062,Senior Product Engineer (Fullstack),Clipster,Software Development,"[backend, frontend, fullstack, golang, postgre...",full_time,2026-07-13 07:05:10,"Europe, UK, Germany, France, European timezones",,NaN,year,NaN,NaN
